# DATA209 — Advanced Exploratory Data Analysis
# Practical P17-18 · Missing values and imputation

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 9 · Module 3 · CO3

---

**Objective.** Compare imputation strategies against a known ground truth and measure what each one does to the variance and shape of the data.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.

This session also reads `pima.csv` to demonstrate real disguised missingness. It is optional — the cell is skipped if the file is absent.

### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 and P15-16 — the dataset and the de-duplicated, standardised copy.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16: duplicates removed, labels standardised
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P17-18 — Missing values and imputation

### A controlled experiment

The shoppers dataset has no missing values, which is an opportunity rather than a problem: we can
**delete values ourselves and keep the truth**. That lets us measure imputation error directly,
which is impossible on real missing data.

We inject two mechanisms:

- **MCAR** — values removed completely at random.
- **MAR** — values removed with a probability that depends on *another observed* column.

Compare with the real-world case in `pima.csv`, where zeros in `Glucose`, `BloodPressure`,
`SkinThickness`, `Insulin` and `BMI` are disguised missing values that `isna()` cannot see.

In [ ]:
# ---- Inject missingness with a known ground truth ----------------------
rng = np.random.RandomState(RANDOM_STATE)

work  = df_clean.copy()
truth = work[["ProductRelated_Duration", "BounceRates", "PageValues"]].copy()

# 1 — MCAR: 15% of ProductRelated_Duration removed at random
mcar_mask = rng.rand(len(work)) < 0.15
work.loc[mcar_mask, "ProductRelated_Duration"] = np.nan

# 2 — MAR: BounceRates missing more often for short sessions
p_missing = np.where(work["ProductRelated"] <= 3, 0.35, 0.05)
mar_mask  = rng.rand(len(work)) < p_missing
work.loc[mar_mask, "BounceRates"] = np.nan

# 3 — MNAR-flavoured: high PageValues hidden more often
p_hide     = np.where(truth["PageValues"] > truth["PageValues"].quantile(0.9), 0.5, 0.03)
mnar_mask  = rng.rand(len(work)) < p_hide
work.loc[mnar_mask, "PageValues"] = np.nan

summary = pd.DataFrame({
    "column"    : ["ProductRelated_Duration", "BounceRates", "PageValues"],
    "mechanism" : ["MCAR", "MAR", "MNAR"],
    "missing"   : [mcar_mask.sum(), mar_mask.sum(), mnar_mask.sum()],
    "missing_%" : [mcar_mask.mean()*100, mar_mask.mean()*100, mnar_mask.mean()*100],
})
print(summary.round(2).to_string(index=False))
print("\nTotal nulls now:", int(work.isna().sum().sum()))

In [ ]:
# ---- Detect and diagnose the mechanism ---------------------------------
# Build an indicator per affected column, then explore it like any other variable.
for c in ["ProductRelated_Duration", "BounceRates", "PageValues"]:
    work[c + "_missing"] = work[c].isna().astype(int)

print("Correlation of each missingness indicator with the OBSERVED columns")
probe = ["ProductRelated", "Administrative", "ExitRates", "SpecialDay"]
diag = pd.DataFrame({
    c: work[probe].corrwith(work[c + "_missing"])
    for c in ["ProductRelated_Duration", "BounceRates", "PageValues"]
})
print(diag.round(3).to_string())

print("\nReading the table")
print("- ProductRelated_Duration: no meaningful correlation -> consistent with MCAR.")
print("- BounceRates: correlates with ProductRelated -> MAR, and ProductRelated is the driver.")
print("- PageValues: little correlation with observed columns, yet we know the missingness")
print("  depends on the hidden value itself. This is exactly why MNAR cannot be diagnosed")
print("  from the data alone — it needs domain knowledge.")

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, c in zip(axes, ["ProductRelated_Duration", "BounceRates", "PageValues"]):
    grp = work.groupby(c + "_missing")["ProductRelated"].median()
    grp.plot(kind="bar", ax=ax, color=["#C9D4DB", "#A6752C"])
    ax.set_title(f"{c}\nmedian ProductRelated by missingness"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Compare imputation strategies

Five strategies on the same column, judged on two criteria: **how close the filled values are to
the truth**, and **what happened to the variance**.

In [ ]:
# ---- Compare strategies against the known truth -------------------------
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

target_col   = "ProductRelated_Duration"      # the MCAR column
support_cols = ["Administrative", "Informational", "ProductRelated",
                "ExitRates", "SpecialDay"]

observed = work[target_col]
actual   = truth[target_col]
mask     = observed.isna().values

strategies = {
    "drop rows"     : None,
    "mean"          : SimpleImputer(strategy="mean"),
    "median"        : SimpleImputer(strategy="median"),
    "KNN (k=5)"     : KNNImputer(n_neighbors=5),
    "iterative"     : IterativeImputer(max_iter=10, random_state=RANDOM_STATE),
}

block   = work[[target_col] + support_cols]
results = []

for name, imp in strategies.items():
    if imp is None:
        kept = observed.dropna()
        results.append({"strategy": name, "n": len(kept), "mean": kept.mean(),
                        "std": kept.std(), "skew": kept.skew(),
                        "MAE_vs_truth": np.nan, "rows_lost": int(mask.sum())})
        continue
    filled = pd.DataFrame(imp.fit_transform(block), columns=block.columns)[target_col]
    mae = np.abs(filled.values[mask] - actual.values[mask]).mean()
    results.append({"strategy": name, "n": len(filled), "mean": filled.mean(),
                    "std": filled.std(), "skew": filled.skew(),
                    "MAE_vs_truth": mae, "rows_lost": 0})

comp = pd.DataFrame(results).set_index("strategy")
comp.loc["TRUE VALUES"] = [len(actual), actual.mean(), actual.std(),
                           actual.skew(), 0.0, 0]
print(comp.round(3).to_string())

### Analyze variance change

Mean imputation gives every missing row the identical value. Those rows then contribute **zero
deviation** from the mean, so the standard deviation must fall. The dataset looks more precise
than it is — confidence intervals narrow and correlations weaken.

In [ ]:
# ---- Quantify the damage ------------------------------------------------
true_std = actual.std()
change = pd.DataFrame({
    "std"        : comp["std"],
    "vs_truth_%" : ((comp["std"] - true_std) / true_std * 100),
    "MAE"        : comp["MAE_vs_truth"],
}).drop(index="TRUE VALUES")
print(change.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

change["vs_truth_%"].plot(kind="bar", ax=axes[0],
                          color=["#8B9199", "#A6752C", "#A6752C", "#1F6F6B", "#1F6F6B"])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("Change in standard deviation vs. the truth (%)"); axes[0].set_xlabel("")

sns.kdeplot(actual, ax=axes[1], label="true values", lw=2, color="black")
for name, imp in [("mean", SimpleImputer(strategy="mean")),
                  ("median", SimpleImputer(strategy="median")),
                  ("KNN (k=5)", KNNImputer(n_neighbors=5))]:
    f = pd.DataFrame(imp.fit_transform(block), columns=block.columns)[target_col]
    sns.kdeplot(f, ax=axes[1], label=name, lw=1.4)
axes[1].set_xlim(0, actual.quantile(0.97))
axes[1].legend(); axes[1].set_title("Distribution after imputation")
plt.tight_layout(); plt.show()

print("Interpretation")
print("- Mean and median imputation shrink the spread and add a spike at the fill value.")
print("- KNN and iterative imputation preserve the shape far better because they use the")
print("  other columns rather than a single constant.")
print("- 'Drop rows' preserves the distribution exactly but discards data — and is only")
print("  unbiased when the mechanism is genuinely MCAR.")

In [ ]:
# ---- The real-world case: disguised missingness in Pima ----------------
pima_path = find("pima.csv") or find("diabetes.csv")
if pima_path:
    pima = pd.read_csv(pima_path)
    zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    zero_cols = [c for c in zero_cols if c in pima.columns]

    print("pandas reports nulls:", int(pima.isna().sum().sum()))
    disguised = pd.DataFrame({
        "zeros"  : (pima[zero_cols] == 0).sum(),
        "zeros_%": (pima[zero_cols] == 0).mean().mul(100).round(1),
    })
    print("\nValues recorded as zero — physiologically impossible:")
    print(disguised.to_string())

    p = pima.copy()
    p[zero_cols] = p[zero_cols].replace(0, np.nan)
    print("\nAfter replacing sentinels with NaN, true missingness:")
    print(p[zero_cols].isna().mean().mul(100).round(1).to_string())
    print(f"\nInsulin std: observed {p['Insulin'].std():.2f} -> "
          f"after mean imputation {p['Insulin'].fillna(p['Insulin'].mean()).std():.2f}")
else:
    print("pima.csv not found — skipping the disguised-missingness demonstration.")

### Deliverable — P17-18

A notebook containing the injected-missingness experiment, the strategy comparison table with
MAE against the truth, the variance-change chart, and a **three-sentence justification** of the
strategy you would adopt and the mechanism you are assuming.